In [61]:
import pandas as pd
import numpy as np

In [62]:
cd C:\Users\tr432\Downloads\Metabolomics_GutLungAxis_re-pipeline

C:\Users\tr432\Downloads\Metabolomics_GutLungAxis_re-pipeline


In [63]:
STYPE = 'serum_pos'
WEEK = '2w'

In [64]:
df1 = pd.read_excel('dataset_filtered_scaled_merged.xlsx', sheet_name=f'{STYPE}_{WEEK}', skiprows=[1])  ## 여기에 원본 Compounds 파일 넣고
df1 = df1.rename(columns={'Label': 'FeatureID'})
# 1) 첫 컬럼명을 FeatureID로
df1 = df1.rename(columns={df1.columns[0]: "FeatureID"})
# 2) 2행(=그룹 라벨 행) 제거: FeatureID 값이 'Group' 또는 'Label'인 행
mask = df1["FeatureID"].astype(str).str.strip().str.lower().isin(["group", "label"])
df1 = df1.loc[~mask].reset_index(drop=True)

df2 = pd.read_excel('stats_merged.xlsx', sheet_name=f'{STYPE}_{WEEK}')  ## 여기엔 FDR로 필터링한 파일 넣고

## 매칭용 이름 정리
df1["Name_key"] = df1["FeatureID"].astype(str).str.strip()
df2["Name_key"] = df2["FeatureID"].astype(str).str.strip()

df1

,FeatureID,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,C10_2w,C11_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key
0,61.03969-1.062,-0.094717,-0.055197,-0.281101,-0.227372,-0.278655,-0.130104,-0.314357,-0.234331,-0.154721,0.252132,0.481365,0.759873,0.484003,-0.100466,-0.106352,61.03969-1.062
1,496.33995-14.389,0.228524,0.180054,0.112347,0.130613,-0.047752,0.023175,-0.183702,0.191706,0.097687,0.215912,0.124215,-1.105425,-0.175320,0.090608,0.117358,496.33995-14.389
2,172.07169-4.382,-1.710834,-1.710834,-1.710834,-1.710834,-1.710834,-1.710834,0.723907,-1.710834,-1.710834,2.558083,2.579500,0.893768,2.157942,2.554414,2.219058,172.07169-4.382
3,520.34-13.811,0.376894,0.199344,0.044692,0.050309,-0.155266,-0.113773,-0.233722,0.151867,0.157459,0.290471,0.087314,-1.211706,-0.286099,0.326146,0.316069,520.34-13.811
4,73.06478-1.434,0.341940,0.668576,0.656013,-3.978812,0.352233,0.764370,0.713451,0.730808,0.796690,0.535783,0.690683,-3.978812,0.663416,0.479125,0.564537,73.06478-1.434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1668,363.27442-15.162,0.951327,-1.810144,1.210122,0.887192,0.890237,0.055315,0.899527,-1.810144,1.149953,1.191684,0.887360,-1.810144,-1.810144,0.928002,-1.810144,363.27442-15.162
1669,316.18052-7.669,-1.045299,1.785516,-1.045299,0.912251,-1.045299,-1.045299,1.620079,1.959608,1.626873,-1.045299,-1.045299,-1.045299,-1.045299,1.503361,-1.045299,316.18052-7.669
1670,290.8553-0.999,-1.067158,0.399323,0.072491,-0.670254,0.445722,0.022946,0.596807,-0.301786,0.295426,-0.340109,0.111805,0.329466,-0.063727,0.373639,-0.204591,290.8553-0.999
1671,361.88999-0.954,0.227399,-0.031577,0.184948,-0.199641,0.308517,0.133646,0.414410,0.034951,0.110664,-0.189330,0.314281,-0.478835,0.042560,0.275603,-1.147596,361.88999-0.954


In [65]:
# df2에서 FeatureID와 Name 같이 가져오기 (Name 없으면 에러)
if "Name" not in df2.columns:
    raise ValueError("df2에 Name 컬럼이 없습니다. df2에서 Name 컬럼명을 확인하세요.")
if "q-val_BKY" not in df2.columns:
    raise ValueError(f"df2에 q-val_BKY 컬럼이 없습니다. 현재 컬럼: {list(df2.columns)}")

# 매칭 키 정리(공백 제거)
df1["FeatureID"] = df1["FeatureID"].astype(str).str.strip()
df2["FeatureID"] = df2["FeatureID"].astype(str).str.strip()

# q-val_BKY 숫자화 후 유의한 FeatureID만 남기기
df2["q-val_BKY"] = pd.to_numeric(df2["q-val_BKY"], errors="coerce")

df2_sig = (
    df2.loc[df2["q-val_BKY"] < 0.05, ["FeatureID", "Name"]]
      .dropna(subset=["FeatureID"])
      .drop_duplicates(subset=["FeatureID"], keep="first")
)

# df1에 df2_sig만 inner join (== 유의한 FeatureID만 매칭)
df1_overlap = pd.merge(
    df1,
    df2_sig,
    on="FeatureID",
    how="inner"
)

print("df1:", df1.shape, "df2_sig:", df2_sig.shape, "overlap:", df1_overlap.shape)
df1_overlap

df1: (1673, 17) df2_sig: (400, 2) overlap: (400, 18)


,FeatureID,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,C10_2w,C11_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key,Name
0,61.03969-1.062,-0.094717,-0.055197,-0.281101,-0.227372,-0.278655,-0.130104,-0.314357,-0.234331,-0.154721,0.252132,0.481365,0.759873,0.484003,-0.100466,-0.106352,61.03969-1.062,Urea
1,172.07169-4.382,-1.710834,-1.710834,-1.710834,-1.710834,-1.710834,-1.710834,0.723907,-1.710834,-1.710834,2.558083,2.579500,0.893768,2.157942,2.554414,2.219058,172.07169-4.382,Metronidazole
2,132.1019-1.822,-0.135217,-0.035935,-0.430127,-0.516225,-0.454265,-0.107077,-0.392276,-0.226497,-0.181825,0.238944,0.335525,0.455272,0.616968,0.458766,0.373970,132.1019-1.822,Isoleucine
3,86.09632-1.816,-0.158929,-0.026130,-0.425810,-0.516255,-0.455543,-0.101578,-0.382745,-0.236495,-0.162148,0.230345,0.343494,0.453695,0.612396,0.446316,0.379388,86.09632-1.816,2-Methylpyrrolidine
4,202.17999-1.496,0.317428,-0.262000,0.084998,-0.406787,0.167104,-0.359711,-0.399592,-0.372489,-0.451668,0.506576,0.289032,0.172459,0.493548,0.371724,-0.150623,202.17999-1.496,11-Aminoundecanoic acid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,138.90647-0.92,0.193503,-0.277185,0.043137,-0.067657,-0.095888,0.210737,0.470314,0.576427,0.469260,-0.100605,-0.127903,-0.390842,-0.164834,-0.202577,-0.535888,138.90647-0.92,NaN
396,582.95841-9.874,-0.209322,-0.275293,-0.108489,-0.292256,0.161148,-1.161944,-0.437989,-0.010144,-0.750054,0.824978,0.732905,0.402822,0.160629,0.632346,0.330661,582.95841-9.874,NaN
397,339.18336-7.177,0.508885,-0.815278,0.023000,-1.439841,0.518977,-0.130698,-1.408451,-1.664817,-0.414253,1.134347,0.626881,1.100002,0.772623,0.591394,0.597231,339.18336-7.177,Benzylviologen
398,421.87911-0.988,-0.019936,-0.188610,0.076301,0.227049,0.226831,-1.169478,-0.121135,-0.072133,-0.692337,0.070260,0.241109,0.388196,0.227219,0.612972,0.193689,421.87911-0.988,NaN


In [66]:
# -----------------------------
# 4) 주차 컬럼 선택 + C/V 샘플만
# -----------------------------
wk = f'_{WEEK}'
group_area_cols = [c for c in df1_overlap.columns if wk in str(c)]
group_area_cols = [c for c in group_area_cols if str(c).strip().startswith(("C","V"))]
print("selected sample cols:", len(group_area_cols))
group_area_cols

selected sample cols: 15


['C03_2w',
 'C04_2w',
 'C05_2w',
 'C06_2w',
 'C07_2w',
 'C08_2w',
 'C09_2w',
 'C10_2w',
 'C11_2w',
 'V05_2w',
 'V06_2w',
 'V07_2w',
 'V08_2w',
 'V09_2w',
 'V10_2w']

In [67]:
# (추가) GroupArea 컬럼들 숫자화
df1_overlap[group_area_cols] = (
    df1_overlap[group_area_cols]
      .replace(r'^\s*$', np.nan, regex=True)     # 빈칸/공백 -> NaN
      .apply(lambda s: pd.to_numeric(s, errors='coerce'))
)

# -----------------------------
# 5) Name 중복 제거: (선택 주차) 평균이 큰 1개만
# -----------------------------
df1_overlap["GroupArea_mean"] = df1_overlap[group_area_cols].mean(axis=1, skipna=True)

df1_overlap = df1_overlap.sort_values(by=["Name", "GroupArea_mean"], ascending=[True, False])
df1_overlap_unique = df1_overlap.drop_duplicates(subset=["Name"], keep="first")

df1_overlap_unique

,FeatureID,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,C10_2w,C11_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key,Name,GroupArea_mean
337,516.31457-8.622,0.288714,-0.498065,-0.253310,-0.561625,-0.136136,-0.316261,-0.374357,-0.167673,-0.338077,0.709593,0.410969,0.419890,0.609572,0.389746,-0.182978,516.31457-8.622,"(11R,23R)-14,17,20-Trihydroxy-23-(octanoyloxy)...",2.059464e-15
162,255.12273-13.274,0.092174,-0.274129,-0.140985,-0.685989,-0.063797,-0.599858,-0.495068,-0.517693,-0.578238,0.868623,0.540250,0.481249,0.970093,0.791456,-0.388088,255.12273-13.274,"(1S)-1,5-Anhydro-1-benzyl-D-galactitol",-3.145632e-16
142,99.04393-10.789,0.170092,-0.337297,-0.081135,-0.521884,-0.014814,-0.403675,-0.387113,-0.383519,-0.627615,0.964021,0.062986,0.331385,0.690529,0.391715,0.146326,99.04393-10.789,"(2E)-Penta-2,4-dienoic acid",-8.141636e-16
27,394.27036-7.387,0.059775,-0.015479,-0.021391,-0.563027,0.248521,-0.461448,-0.593240,-0.556024,-0.320948,0.733715,0.308657,0.528579,0.554175,0.254685,-0.156549,394.27036-7.387,"(2S,3R,4E)-2-Amino-3-hydroxy-4-nonadecen-1-yl ...",-1.480297e-17
172,127.11172-7.812,-0.245681,0.694873,0.916755,0.200489,0.478177,0.544451,0.229386,0.764374,0.627657,-1.484802,-0.981314,-1.438904,-0.351784,-0.286313,0.332638,127.11172-7.812,"(2Z,4E)-2,4-Octadien-1-ol",6.772360e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,146.06009-5.736,-0.340424,-0.512564,-0.126765,-0.272407,-0.229403,-0.173542,0.011277,-0.283285,-0.060242,0.762969,0.195439,0.175609,0.384152,0.395483,0.073704,146.06009-5.736,quinolone,2.118676e-16
147,382.272-13.21,-0.041746,0.023877,0.203744,-0.389364,-0.562842,-0.180355,-0.543995,-0.246454,-0.348212,0.110482,0.792191,-0.083692,0.677149,0.105001,0.484216,382.272-13.21,sphinganine 1-phosphate,-9.214851e-16
17,203.08122-1.471,0.178341,-0.144290,0.100638,-0.381594,0.138717,-0.364861,-0.323195,-0.328814,-0.543352,0.409594,0.364775,0.232828,0.452851,0.239995,-0.031633,203.08122-1.471,tolimidone,2.971697e-15
7,188.07063-5.736,-0.519445,-0.370204,-0.267136,-0.190289,-0.286647,-0.181128,-0.090352,-0.277298,-0.143758,0.782358,0.418351,0.216479,0.388096,0.421725,0.099249,188.07063-5.736,trans-3-Indoleacrylic acid,-6.448545e-16


In [68]:
# -----------------------------
# 6) 저장은 Name으로(FeatureID는 매칭용으로만 사용)
# -----------------------------
result = df1_overlap_unique[["Name"] + group_area_cols].copy()

# 2행(Group) 추가 + 1열 헤더를 Label로 바꾸기(원하면)
out = result.rename(columns={"Name":"Label"}).copy()

def to_group(c):
    s = str(c).strip()
    if s.startswith("C"): return "Control"  #Edit
    if s.startswith("V"): return "VNAM"  #Edit
    return "NA"

group_row = ["Group"] + [to_group(c) for c in group_area_cols]
group_df = pd.DataFrame([group_row], columns=["Label"] + group_area_cols)

out2 = pd.concat([group_df, out], ignore_index=True)
out2.to_csv(f'MA_PW_{WEEK}_sig.csv', index=False, encoding="utf-8-sig")

print(out2.head(3))
print(out2.shape)

                                               Label    C03_2w    C04_2w  \
0                                              Group   Control   Control   
1  (11R,23R)-14,17,20-Trihydroxy-23-(octanoyloxy)...  0.288714 -0.498065   
2             (1S)-1,5-Anhydro-1-benzyl-D-galactitol  0.092174 -0.274129   

     C05_2w    C06_2w    C07_2w    C08_2w    C09_2w    C10_2w    C11_2w  \
0   Control   Control   Control   Control   Control   Control   Control   
1  -0.25331 -0.561625 -0.136136 -0.316261 -0.374357 -0.167673 -0.338077   
2 -0.140985 -0.685989 -0.063797 -0.599858 -0.495068 -0.517693 -0.578238   

     V05_2w    V06_2w    V07_2w    V08_2w    V09_2w    V10_2w  
0      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM  
1  0.709593  0.410969   0.41989  0.609572  0.389746 -0.182978  
2  0.868623   0.54025  0.481249  0.970093  0.791456 -0.388088  
(82, 16)
